## Step 4: Delivery — Cortex Analyst Chat Interface (Fixed)

**Project:** AI-Powered Sales Data Insights with Snowflake + Cortex Analyst

This fixed notebook removes the failing `SNOWFLAKE.CORTEX.ANALYST(...)` SQL function call. Your Snowflake account returned `Unknown user-defined function SNOWFLAKE.CORTEX.ANALYST`, so the working implementation should call Cortex Analyst through the REST API from Streamlit in Snowflake.

**Run order:**
1. Verify curated tables
2. Create/verify the `SEMANTIC_MODELS` stage
3. Verify `sales_analytics_semantic_model.yaml` is uploaded
4. Sanity-check curated KPIs
5. Deploy the fixed Streamlit app

**Important:** Do not run task automation until the chat app works. The chat does not need tasks to run.


In [ ]:
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()

try:
    session.query_tag = {
        "origin": "raj_manala",
        "name": "sales_analytics_ai_insights",
        "version": {"major": 1, "minor": 1},
        "attributes": {"step": "delivery_cortex_analyst_fixed"}
    }
except Exception:
    pass

print(f"Session ready. Database: {session.get_current_database()}, Schema: {session.get_current_schema()}")

In [ ]:
DATABASE_NAME = "SALES_ANALYTICS_2025"
RAW_SCHEMA = "RAW_DATA"
CURATED_SCHEMA = "CURATED"
STAGE_NAME = "SEMANTIC_MODELS"
YAML_FILE = "sales_analytics_semantic_model.yaml"
SEMANTIC_MODEL_PATH = f"@{DATABASE_NAME}.{RAW_SCHEMA}.{STAGE_NAME}/{YAML_FILE}"

print("Semantic model path:", SEMANTIC_MODEL_PATH)

### 1. Verify the curated tables from Step 2 and Step 3

Run this first. If `FACT_SALES_ENRICHED` is missing, go back and rerun Step 2.

In [ ]:
USE DATABASE SALES_ANALYTICS_2025;

SELECT TABLE_SCHEMA, TABLE_NAME, ROW_COUNT
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA IN ('CURATED', 'RAW_DATA')
ORDER BY TABLE_SCHEMA, TABLE_NAME;

### 2. Verify the main fact table has the columns the YAML expects

In [ ]:
DESC TABLE SALES_ANALYTICS_2025.CURATED.FACT_SALES_ENRICHED;

### 3. Sanity-check the main business metrics

This proves your data layer is ready before connecting the chat interface.

In [ ]:
SELECT 
    COUNT(*) AS total_rows,
    ROUND(SUM(sales_amount), 2) AS total_revenue,
    ROUND(SUM(gross_profit), 2) AS total_profit,
    ROUND(AVG(profit_margin_pct), 2) AS avg_margin_pct,
    ROUND(AVG(shipping_days), 2) AS avg_shipping_days
FROM SALES_ANALYTICS_2025.CURATED.FACT_SALES_ENRICHED;

### 4. Create/verify the internal stage for the semantic model YAML

This is a normal Snowflake internal stage. It does not need to be a special semantic stage.

In [ ]:
USE DATABASE SALES_ANALYTICS_2025;
USE SCHEMA RAW_DATA;

CREATE STAGE IF NOT EXISTS SEMANTIC_MODELS
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'Stage for Cortex Analyst semantic model YAML files';

SHOW STAGES LIKE 'SEMANTIC_MODELS';

### 5. Upload the YAML file

Use whichever option works in your Snowflake setup.

**Option A — Snowsight UI**
1. Go to **Data / Catalog → Databases → SALES_ANALYTICS_2025 → RAW_DATA → Stages → SEMANTIC_MODELS**
2. Click **+ Files**
3. Upload `sales_analytics_semantic_model.yaml`

**Option B — SnowSQL / Snowflake CLI**
```sql
PUT file:///Users/raj/Downloads/sales_analytics_semantic_model.yaml
@SALES_ANALYTICS_2025.RAW_DATA.SEMANTIC_MODELS
AUTO_COMPRESS = FALSE
OVERWRITE = TRUE;
```

**Option C — Notebook workspace upload**
Use the next cell only if the YAML file is present in this notebook's file workspace.

In [ ]:
# Optional: upload YAML from the notebook workspace.
# If this fails, it is okay. Use Snowsight upload or SnowSQL PUT instead.

try:
    put_result = session.file.put(
        YAML_FILE,
        f"@{DATABASE_NAME}.{RAW_SCHEMA}.{STAGE_NAME}",
        auto_compress=False,
        overwrite=True,
    )
    print("Semantic model upload attempted from notebook workspace. Result:")
    for row in put_result:
        print(row)
except Exception as e:
    print("Notebook upload did not run successfully. This is okay if you already uploaded through Snowsight.")
    print("Reason:", e)

### 6. Verify the YAML file is in the stage

This is the key Step 4 checkpoint. If this cell shows `sales_analytics_semantic_model.yaml`, the semantic model file is ready.

In [ ]:
LS @SALES_ANALYTICS_2025.RAW_DATA.SEMANTIC_MODELS;

### 7. Cortex Analyst test — fixed

The old notebook used this and failed:

```python
SELECT SNOWFLAKE.CORTEX.ANALYST(...)
```

That SQL function is not available in your account, so this notebook no longer uses it.

Cortex Analyst is meant to be called through the REST API endpoint `/api/v2/cortex/analyst/message`. In Streamlit in Snowflake, the supported way is `_snowflake.send_snow_api_request(...)`.

The helper below will not crash the notebook. If `_snowflake` is not available inside your notebook runtime, it will clearly tell you to continue with the Streamlit app. That is expected.

In [ ]:
API_ENDPOINT = "/api/v2/cortex/analyst/message"

try:
    import _snowflake
except Exception:
    _snowflake = None


def ask_cortex_analyst(question: str, run_generated_sql: bool = True):
    """Test Cortex Analyst through the Snowflake REST API when available.

    Note: `_snowflake.send_snow_api_request` is available in Streamlit in Snowflake.
    Some Snowflake notebook runtimes may not expose `_snowflake`. If it is not
    available here, skip this notebook test and deploy the fixed Streamlit app.
    """

    if _snowflake is None:
        print("_snowflake is not available in this notebook runtime.")
        print("This is not a data/YAML problem. Continue with the fixed Streamlit app.")
        print("The app uses the same REST endpoint: /api/v2/cortex/analyst/message")
        return None

    request_body = {
        "messages": [
            {
                "role": "user",
                "content": [{"type": "text", "text": question}],
            }
        ],
        "semantic_model_file": SEMANTIC_MODEL_PATH,
    }

    resp = _snowflake.send_snow_api_request(
        "POST",
        API_ENDPOINT,
        {},
        {},
        request_body,
        {},
        30000,
    )

    if resp.get("status", 500) >= 400:
        print("Cortex Analyst API request failed. Full response:")
        print(resp)
        return resp

    response = json.loads(resp["content"])
    content = response.get("message", {}).get("content", [])

    for item in content:
        if item.get("type") == "text":
            print("Answer:")
            print(item.get("text", ""))
        elif item.get("type") == "suggestions":
            print("Suggested follow-ups:")
            for suggestion in item.get("suggestions", []):
                print("-", suggestion)
        elif item.get("type") == "sql":
            sql = item.get("statement", "")
            print("Generated SQL:")
            print(sql)
            if run_generated_sql and sql.strip():
                print("Results:")
                session.sql(sql).show()

    return response

In [ ]:
print("=" * 60)
print("Question: What is the total revenue?")
print("=" * 60)
ask_cortex_analyst("What is the total revenue?")

In [ ]:
print("=" * 60)
print("Question: Top 5 products by revenue")
print("=" * 60)
ask_cortex_analyst("Top 5 products by revenue")

In [ ]:
print("=" * 60)
print("Question: Revenue by territory")
print("=" * 60)
ask_cortex_analyst("Revenue by territory")

### 8. Deploy the fixed Streamlit chat app

The working chat interface should be deployed in **Streamlit in Snowflake**, not tested through the old SQL function.

Use the attached file: `streamlit_chat_app_fixed.py`

Deployment steps:
1. Go to **Projects → Streamlit → + Streamlit App**
2. App name: `Sales Analytics AI Chat`
3. Warehouse: `COMPUTE_WH`
4. Database: `SALES_ANALYTICS_2025`
5. Schema: `RAW_DATA`
6. Replace the default app code with `streamlit_chat_app_fixed.py`
7. Click **Run**

Test these questions in the app:
- What is the total revenue?
- Top 5 products by revenue
- Revenue by territory
- Compare Mountain vs Road bikes
- Average shipping days

If the app returns generated SQL and a dataframe, your chat interface is working end-to-end.

### 9. Permissions checklist

If the Streamlit app cannot read the semantic model or tables, run these grants with `ACCOUNTADMIN` after replacing `<YOUR_ROLE>` with your actual role.

```sql
USE ROLE ACCOUNTADMIN;

GRANT DATABASE ROLE SNOWFLAKE.CORTEX_ANALYST_USER TO ROLE <YOUR_ROLE>;

GRANT USAGE ON DATABASE SALES_ANALYTICS_2025 TO ROLE <YOUR_ROLE>;
GRANT USAGE ON SCHEMA SALES_ANALYTICS_2025.RAW_DATA TO ROLE <YOUR_ROLE>;
GRANT USAGE ON SCHEMA SALES_ANALYTICS_2025.CURATED TO ROLE <YOUR_ROLE>;

GRANT READ, WRITE ON STAGE SALES_ANALYTICS_2025.RAW_DATA.SEMANTIC_MODELS TO ROLE <YOUR_ROLE>;
GRANT SELECT ON ALL TABLES IN SCHEMA SALES_ANALYTICS_2025.CURATED TO ROLE <YOUR_ROLE>;
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE <YOUR_ROLE>;
```

To check your current role:

```sql
SELECT CURRENT_ROLE();
```

### 10. Task automation is optional — do not run it yet

Do not create or resume the daily refresh task until the chat app works. For the demo, you only need:

1. Curated tables working
2. YAML uploaded to stage
3. Fixed Streamlit app running

After the demo works, the task should be rebuilt using the full Step 2 transformation logic. Do not use a simplified task that recreates `FACT_SALES_ENRICHED` with fewer columns, because the semantic model expects the enriched columns.

### Step 4 complete checklist

You are ready for the chat app when these are true:

- `FACT_SALES_ENRICHED` exists in `SALES_ANALYTICS_2025.CURATED`
- KPI sanity-check query runs successfully
- `LS @SALES_ANALYTICS_2025.RAW_DATA.SEMANTIC_MODELS;` shows `sales_analytics_semantic_model.yaml`
- The fixed Streamlit app uses `_snowflake.send_snow_api_request(...)`
- The Streamlit app returns generated SQL and real query results from Snowflake

**Final architecture:**

```text
Curated Snowflake Tables
        ↓
Semantic Model YAML on Stage
        ↓
Cortex Analyst REST API
        ↓
Streamlit in Snowflake Chat App
        ↓
Real-time answers + generated SQL + charts
```